# Tools - 工具
工具扩展了智能体的能力，允许它们获取实时数据、执行代码、查询外部数据库以及在世界中采取行动。

在底层，工具就是是具有 **明确定义** 的输入和输出的 **可调用函数** ,
这些输入和输出会被传递给模型。模型根据对话上下文决定何时调用工具，以及提供什么输入参数



## Create Tools

### 基本的Tools创建
创建工具最简单的方法是使用 `@tool` 装饰器。默认情况下，函数的文档字符串（doc-string）成为工具的描述，帮助模型理解何时使用它，会加载到context上下文中。



In [2]:
from langchain.tools import tool

@tool
def search_db(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query (str): Search terms to look for
        limit (int, optional): Maximum number of results to return. Defaults to 10
    """
    return f"Found {limit} results for '{query}'"


【⚠️注意】

定义Tools的时候要进行 **类型提示** 

在定义工具的时候还可以在装饰器里面传入 name 和 description

In [3]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))
#? @tool 的第一个参数是名字，描述传入
# 这里面传入的


## 高级的工具定义

通过类型提示如果遇到复杂的输入格式可能不太行，比如说要指定输入的字面量等等，这是有就要定义复杂的输入格式 schema

有两种实现方式一种是Pydantic的BaseModel，一种是json格式定义

In [ ]:
from pydantic import BaseModel, Field
# BaseModel 是用来继承的
# Field 是定义默认值和描述的
from typing import Literal
# Literal 用于指定字面量

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )
    
#? tool(args_schema = ) 
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

#! args_schema 的类型
# (type) ArgsSchema = type[BaseModel] | dict[str, Any]
# dict也就是json格式

In [ ]:
 #? 使用json格式

weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

#! 使用json格式的话就不能指定units的字面量，只能指定默认值

print(type(weather_schema))

@tool(args_schema=weather_schema)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

<class 'dict'>


tool装饰器的args：

以下参数名称是保留的，不能用作工具参数。使用这些名称会导致运行时错误

- config：保留用于在内部将 RunnableConfig 传递给工具
- runtime：保留用于 ToolRuntime 参数（访问状态、上下文、存储）

如果要访问运行时信息，请使用 ToolRuntime 参数，而不是自行命名参数 config 或 runtime

## Access Context 访问上下文

工具能访问运行时的信息，比如说对话历史、用户数据和持久化记忆，Tools可以通过 `ToolRuntime` 访问运行时信息，这是一个类，传入Tools中，然后再Tools中就可以去访问这个类的属性来达到访问上下文的功能。

ToolRuntime的参数：


| 组件 | 描述 | 应用场景 |
| --- | --- | --- |
| State 状态 | 短期记忆 - 当前对话中存在的可变数据（消息、计数器、自定义字段） | 访问对话历史，跟踪工具调用次数 |
| Context 上下文 | 在调用时传递的不可变配置（用户 ID、会话信息） | 根据用户身份个性化回复 |
| Store 存储 | 长期记忆 - 跨对话持续存在的持久数据 | 保存用户偏好，维护知识库 |
| Stream Writer 流写入器 | 在工具执行期间实时发出更新 | 显示长时间运行操作的进度 |
| Config 配置 | RunnableConfig 用于执行 | 访问回调、标签和元数据 |
| Tool Call ID 工具调用 ID | 当前工具调用的唯一标识符 | 关联日志和模型调用的工具调用 |